In [2]:
! pip install -q kaggle

In [3]:
# prompt: fix above problem

from google.colab import files

files.upload()

# Create a directory for Kaggle configuration files if it doesn't exist
!mkdir -p ~/.kaggle

!cp kaggle.json ~/.kaggle/
# Make the kaggle.json file executable (required by the Kaggle API)
!chmod 600 ~/.kaggle/kaggle.json

#! mkdir ~/.kaggle

!cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle (1).json


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!kaggle competitions download -c superai5-esan-to-thai-machine-translation

superai5-esan-to-thai-machine-translation.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
!unzip superai5-esan-to-thai-machine-translation.zip

Archive:  superai5-esan-to-thai-machine-translation.zip
replace sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: sample_submission.csv   
  inflating: submission.csv          
  inflating: test.csv                
  inflating: train.csv               
  inflating: val.csv                 


In [7]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ["PYTHONWARNINGS"] = "ignore"

In [8]:
!nvidia-smi

Wed Jul  2 22:40:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
!pip install simpletransformers==0.70.1
!pip install transformers==4.51.1

In [9]:
import logging
import pandas as pd
from simpletransformers.seq2seq import Seq2SeqModel, Seq2SeqArgs
import torch, os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.system("rm -rf outputs/ cache_dir/ runs/")

# Setup logging
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# Load and prepare the datasets
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")

# Rename columns
train_df = train_df.rename(columns={"input": "input_text", "output": "target_text"})
val_df = val_df.rename(columns={"input": "input_text", "output": "target_text"})

# Remove '\n' from text
train_df["input_text"] = train_df["input_text"].str.replace("\n", "", regex=False)
train_df["target_text"] = train_df["target_text"].str.replace("\n", "", regex=False)
val_df["input_text"] = val_df["input_text"].str.replace("\n", "", regex=False)
val_df["target_text"] = val_df["target_text"].str.replace("\n", "", regex=False)

train_df = train_df[["input_text", "target_text"]]
val_df = val_df[["input_text", "target_text"]]

full = pd.concat([train_df, val_df])
full

,input_text,target_text
0,คึดฮอดเจ้าหลายเด้อ,คิดถึงเธอมากนะ
1,เจ้าสิไปเบิ่งคอนเสิร์ตหมอลำซิ่งนำกันบ่มื้อนี้แลง,คุณจะไปดูคอนเสิร์ตหมอลำซิ่งด้วยกันไหมเย็นนี้
2,เฮาสิไปเลาะในเมือง,เราจะไปเดินเล่นในเมือง
3,เฮามาเฮ็ดแนวกินนำกัน,เรามาทำอาหารด้วยกัน
4,เมื่อวานเฮาไปฟังหมอลำหน้าวัดม่วนหลายสนุกจนบ่มี...,เมื่อวานเราไปฟังหมอลำหน้าวัดสนุกมากจนไม่มีใครอ...
...,...,...
107,ข่อยคิดฮอดอิแม่แฮง,ฉันคิดถึงแม่มาก
108,ซื้อสองชิ้นลดบ่,ซื้อสองชิ้นลดไหม
109,เงินเดือนขึ้นบ่,เงินเดือนขึ้นไหม
110,ไปเถาะอย่ารอหลาย,ไปเถอะอย่ารอนาน


In [10]:
# Define model arguments
model_args = {
    "overwrite_output_dir": True,
    "num_train_epochs": 5,
    "gradient_accumulation_steps": 2,
    "train_batch_size": 8,
    "max_seq_length": 64,
    'n_gpu': 1,
}

# Initialize the model
model = Seq2SeqModel(
    encoder_decoder_type="mbart",
    encoder_decoder_name="facebook/mbart-large-50-many-to-many-mmt",
    max_length=64,
    args=model_args,
)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizerFast'.


In [11]:
# Train the model
model.train_model(train_df)

  0%|          | 0/1943 [00:00<?, ?it/s]

Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Running Epoch 1 of 5:   0%|          | 0/243 [00:00<?, ?it/s]

Running Epoch 2 of 5:   0%|          | 0/243 [00:00<?, ?it/s]

Running Epoch 3 of 5:   0%|          | 0/243 [00:00<?, ?it/s]

Running Epoch 4 of 5:   0%|          | 0/243 [00:00<?, ?it/s]

Running Epoch 5 of 5:   0%|          | 0/243 [00:00<?, ?it/s]

(605, 1.4880535423509345)

In [12]:
test_df = pd.read_csv("test.csv")
test_df["input"] = test_df["input"].str.replace("\n", "", regex=False)
test_df

,id,input
0,2055,ข่อยบ่มีเวลาหลาย
1,2056,บ่ฮู้ว่าสิใช้แอปนี้จั่งใด
2,2057,เจ้ามีเบอร์โทรข่อยบ่
3,2058,เฮามาสิลองเฮ็ดเบิ่ง
4,2059,ไปนอนก่อนเด้อ
...,...,...
369,2424,กาแฟสิใช้เมล็ดจากบราซิลบ่
370,2425,ถ้าซื้อหลายชิ้นลดเพิ่มได้บ่
371,2426,เจ้าคิดจังได๋
372,2427,เฮ็ดเวียกอยู่บ้อ


In [13]:
!pip install -q pythainlp

In [14]:
from pythainlp.tokenize import word_tokenize

submission = pd.read_csv('sample_submission.csv')
submission['output'] = model.predict(test_df['input'].tolist())
submission['output'] = submission['output'].apply(lambda x: ' '.join(word_tokenize(x, engine="newmm")))
submission.to_csv('submission.csv', index=False)
submission

Generating outputs:   0%|          | 0/4 [00:00<?, ?it/s]

,id,output
0,2055,ฉัน ไม่ มี เวลา มาก
1,2056,ไม่ รู้ ว่า จะ ซื้อ แอ ป นี้ เมื่อไหร่
2,2057,เธอ มี เบอร์ โทร ฉัน ไหม
3,2058,เรา จะ มา ลอง ทํา ดู
4,2059,ไป นอน ก่อน นะ
...,...,...
369,2424,กาแฟ จะ ใช้ เมล็ด จาก บราซิล ไหม
370,2425,ต้อง ซื้อ เยอะ มาก เลย ลด เพิ่ม ได้ ไหม
371,2426,เธอ คิด ยังไง
372,2427,ทํา งาน อยู่ บ ไหม


In [ ]:
#kaggle competitions submit -c superai5-esan-to-thai-machine-translation -f submission.csv -m "Message"

In [15]:
!pip list | grep transformers

sentence-transformers                 4.1.0
simpletransformers                    0.70.1
transformers                          4.51.1


In [16]:
submission.to_csv('submission.csv', index=False)

In [17]:
!kaggle competitions submit -c superai5-esan-to-thai-machine-translation -f submission.csv -m "Message"

100% 27.2k/27.2k [00:00<00:00, 72.3kB/s]
Successfully submitted to ESAN to THAI Machine Translation